# 03 · Silver cleaning (REQ-02)
**Purpose:** Read Bronze as a stream, fix data types, remove bad rows, decode codes into labels, and add useful columns.

In [0]:
%run ./00_config

# Scratch / Practice Notebook

This notebook is my personal scratch space for experimenting and practicing new PySpark/Spark SQL concepts while building this project.

**It is NOT part of the graded pipeline.** The real pipeline is: `00_config` -> `01_setup_landing_zone` -> `02_bronze_ingestion` -> `03_silver_cleaning` -> `04_gold_aggregations` -> `05_analytics` -> `06_incremental_test`.

# 00 · Config
All project names and paths in one place. Every other notebook loads this with `%run ./00_config`.

✅ Config loaded. Landing folder: /Volumes/workspace/nyc_taxi/pipeline_files/landing


In [0]:
bronze = spark.table(BRONZE_TABLE)

# Count rows for each payment_type code
display(bronze.groupBy("payment_type").count().orderBy("payment_type"))

payment_type,count
null,33959
1,36609604
2,13767737
3,260110
4,98115
5,30


In [0]:
# Count rows for each VendorID code
display(bronze.groupBy("VendorID").count().orderBy("VendorID"))

VendorID,count
null,33959
1,18838042
2,31630775
4,266779


In [0]:
from pyspark.sql import functions as F

# Convert only the columns we need from text to real types
check = spark.table(BRONZE_TABLE).select(
    F.to_timestamp("tpep_pickup_datetime").alias("pickup"),
    F.to_timestamp("tpep_dropoff_datetime").alias("dropoff"),
    F.col("trip_distance").cast("double").alias("distance"),
    F.col("fare_amount").cast("double").alias("fare"),
    F.col("passenger_count").cast("int").alias("passengers"),
)

# Trip length in minutes = (dropoff seconds - pickup seconds) / 60
minutes = (F.unix_timestamp("dropoff") - F.unix_timestamp("pickup")) / 60

total = check.count()

# A list of problems: name -> how many rows have it
problems = {
    "distance <= 0":              check.filter(F.col("distance") <= 0).count(),
    "distance > 300":             check.filter(F.col("distance") > 300).count(),
    "dropoff not after pickup":   check.filter(F.col("dropoff") <= F.col("pickup")).count(),
    "fare < 0":                   check.filter(F.col("fare") < 0).count(),
    "passengers = 0":             check.filter(F.col("passengers") == 0).count(),
    "trip > 6 hours":             check.filter(minutes > 360).count(),
    "pickup outside Jan-Mar 2019": check.filter(
        (F.col("pickup") < "2019-01-01") | (F.col("pickup") >= "2019-04-01")
    ).count(),
}

print("Total rows:", f"{total:,}")
for name, n in problems.items():
    print(f"{name:30} {n:>10,}   ({n / total:.2%})")

Total rows: 50,769,555
distance <= 0                     394,920   (0.78%)
distance > 300                          8   (0.00%)
dropoff not after pickup           48,807   (0.10%)
fare < 0                           81,259   (0.16%)
passengers = 0                    902,288   (1.78%)
trip > 6 hours                    133,986   (0.26%)
pickup outside Jan-Mar 2019    28,250,482   (55.64%)


In [0]:
from pyspark.sql import functions as F

# 1) Read Bronze AS A STREAM (each run only gets rows not processed yet)
bronze_stream = spark.readStream.table(BRONZE_TABLE)

# 2) Keep the columns we need, and convert text -> proper types
typed = bronze_stream.select(
    F.col("VendorID").cast("int").alias("vendor_id"),
    F.to_timestamp("tpep_pickup_datetime").alias("pickup_time"),
    F.to_timestamp("tpep_dropoff_datetime").alias("dropoff_time"),
    F.col("passenger_count").cast("int").alias("passenger_count"),
    F.col("trip_distance").cast("double").alias("trip_distance"),
    F.col("PULocationID").cast("int").alias("pickup_zone"),
    F.col("DOLocationID").cast("int").alias("dropoff_zone"),
    F.col("payment_type").cast("int").alias("payment_type"),
    F.col("fare_amount").cast("double").alias("fare_amount"),
    F.col("tip_amount").cast("double").alias("tip_amount"),
    F.col("tolls_amount").cast("double").alias("tolls_amount"),
    F.col("total_amount").cast("double").alias("total_amount"),
    F.col("source_file"),
    F.col("ingested_at"),
)

typed.printSchema()

root
 |-- vendor_id: integer (nullable = true)
 |-- pickup_time: timestamp (nullable = true)
 |-- dropoff_time: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_zone: integer (nullable = true)
 |-- dropoff_zone: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



In [0]:
# Helper column 1: trip length in minutes
# Helper column 2: the month written in the file name, e.g. "2019-01"
with_helpers = (
    typed
        .withColumn("trip_minutes",
                    (F.unix_timestamp("dropoff_time") - F.unix_timestamp("pickup_time")) / 60)
        .withColumn("file_month",
                    F.regexp_extract("source_file", r"(\d{4}-\d{2})", 1))
)

# Keep ONLY rows that pass ALL 6 rules
clean = with_helpers.filter(
    (F.col("trip_distance") > 0)                                           # R1: the taxi moved
    & (F.col("trip_distance") <= 300)                                      # R2: realistic NYC distance
    & (F.col("dropoff_time") > F.col("pickup_time"))                       # R3: time goes forward
    & (F.col("fare_amount") >= 0)                                          # R4: no negative fares
    & (F.col("trip_minutes") <= 360)                                       # R5: trip under 6 hours
    & (F.date_format("pickup_time", "yyyy-MM") == F.col("file_month"))     # R6: date matches its file
)

print("✅ Cleaning rules ready")

✅ Cleaning rules ready


In [0]:
enriched = (
    clean
        # ----- New helpful columns -----
        .withColumn("pickup_date", F.to_date("pickup_time"))
        .withColumn("pickup_hour", F.hour("pickup_time"))
        .withColumn("day_type",
                    F.when(F.dayofweek("pickup_time").isin(1, 7), "Weekend")
                     .otherwise("Weekday"))
        .withColumn("tip_percent",
                    F.when(F.col("fare_amount") > 0,
                           F.round(F.col("tip_amount") / F.col("fare_amount") * 100, 2)))
        .withColumn("avg_speed_mph",
                    F.round(F.col("trip_distance") / (F.col("trip_minutes") / 60), 2))

        # ----- Codes -> readable labels -----
        .withColumn("payment_method",
                    F.when(F.col("payment_type") == 1, "Credit card")
                     .when(F.col("payment_type") == 2, "Cash")
                     .when(F.col("payment_type") == 3, "No charge")
                     .when(F.col("payment_type") == 4, "Dispute")
                     .otherwise("Unknown"))
        .withColumn("vendor_name",
                    F.when(F.col("vendor_id") == 1, "Creative Mobile Technologies")
                     .when(F.col("vendor_id") == 2, "VeriFone")
                     .otherwise("Unknown vendor"))

        # The helper column file_month was only needed for rule R6
        .drop("file_month")
)

enriched.printSchema()

root
 |-- vendor_id: integer (nullable = true)
 |-- pickup_time: timestamp (nullable = true)
 |-- dropoff_time: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_zone: integer (nullable = true)
 |-- dropoff_zone: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- trip_minutes: double (nullable = true)
 |-- pickup_date: date (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- day_type: string (nullable = false)
 |-- tip_percent: double (nullable = true)
 |-- avg_speed_mph: double (nullable = true)
 |-- payment_method: string (nullable = false)
 |-- vendor_name: string (nullable = false)



In [0]:
%run ./00_config

# Scratch / Practice Notebook

This notebook is my personal scratch space for experimenting and practicing new PySpark/Spark SQL concepts while building this project.

**It is NOT part of the graded pipeline.** The real pipeline is: `00_config` -> `01_setup_landing_zone` -> `02_bronze_ingestion` -> `03_silver_cleaning` -> `04_gold_aggregations` -> `05_analytics` -> `06_incremental_test`.

# 00 · Config
All project names and paths in one place. Every other notebook loads this with `%run ./00_config`.

✅ Config loaded. Landing folder: /Volumes/workspace/nyc_taxi/pipeline_files/landing


In [0]:
query = (
    enriched.writeStream
        .format("delta")
        .option("checkpointLocation", f"{CHECKPOINT_BASE}/silver")   # Silver's OWN notebook 📒
        .trigger(availableNow=True)
        .toTable(SILVER_TABLE)
)

query.awaitTermination()
print("✅ Silver finished")

✅ Silver finished


In [0]:
bronze_rows = spark.table(BRONZE_TABLE).count()
silver_rows = spark.table(SILVER_TABLE).count()
removed = bronze_rows - silver_rows

print(f"Bronze rows:  {bronze_rows:,}")
print(f"Silver rows:  {silver_rows:,}")
print(f"Removed:      {removed:,}  ({removed / bronze_rows:.2%})")

Bronze rows:  50,769,555
Silver rows:  50,169,175
Removed:      600,380  (1.18%)


In [0]:
display(
    spark.table(SILVER_TABLE)
        .select("pickup_time", "pickup_hour", "day_type", "trip_minutes",
                "trip_distance", "payment_method", "tip_percent", "vendor_name")
        .limit(10)
)

pickup_time,pickup_hour,day_type,trip_minutes,trip_distance,payment_method,tip_percent,vendor_name
2019-05-01T00:35:54.000Z,0,Weekday,1.55,1.5,Cash,0.0,Creative Mobile Technologies
2019-05-01T00:37:45.000Z,0,Weekday,0.06666666666666667,1.5,Cash,0.0,Creative Mobile Technologies
2019-05-01T00:44:57.000Z,0,Weekday,5.233333333333333,0.7,Cash,0.0,Creative Mobile Technologies
2019-05-01T00:59:48.000Z,0,Weekday,10.566666666666666,2.0,Credit card,21.05,Creative Mobile Technologies
2019-05-01T00:23:20.000Z,0,Weekday,9.616666666666667,2.5,Cash,0.0,Creative Mobile Technologies
2019-05-01T00:35:54.000Z,0,Weekday,11.916666666666666,2.8,Cash,0.0,Creative Mobile Technologies
2019-05-01T00:20:12.000Z,0,Weekday,6.666666666666667,2.0,Credit card,29.38,Creative Mobile Technologies
2019-05-01T00:34:34.000Z,0,Weekday,13.883333333333333,3.7,Credit card,25.56,Creative Mobile Technologies
2019-05-01T00:07:15.000Z,0,Weekday,12.433333333333334,2.5,Credit card,26.82,Creative Mobile Technologies
2019-05-01T00:16:48.000Z,0,Weekday,10.816666666666666,2.5,Credit card,27.5,Creative Mobile Technologies


In [0]:
display(
    spark.table(SILVER_TABLE)
        .groupBy("pickup_hour")
        .count()
        .orderBy("pickup_hour")
)

pickup_hour,count
0,1466124
1,1016016
2,704588
3,496851
4,387157
5,462667
6,1076262
7,1852510
8,2323608
9,2349195


### Data quality rules (a row is removed if it breaks any rule)
| # | Rule (keep only if…) | Why | Rows failing (Jan–Mar 2019) |
|---|---|---|---|
| R1 | distance > 0 | A real ride must move; 0 or negative = cancelled trip or meter error | 157,809 |
| R2 | distance ≤ 300 miles | Longer is unrealistic for NYC (831 mi ≈ NYC→Chicago) | 3 |
| R3 | dropoff after pickup | Time cannot go backwards | 18,455 |
| R4 | fare ≥ 0 | Negative fares are refunds or corrections, not rides | 28,380 |
| R5 | trip ≤ 6 hours | Longer trips mean the meter was left running | 59,063 |
| R6 | pickup month = file month | Catches wrong dates (e.g. 2008); works for any future month | ~800 |

**Result:** 242,168 of 22,519,712 rows removed (**1.08%**). Some rows broke several rules, so the total is less than the sum.

**Kept on purpose:**
- 0-passenger rides (376,662): passenger count is typed by the driver and often skipped; the rides are real and passenger count isn't needed for our questions.
- VendorID 4 (170,407): not in the data dictionary; labelled "Unknown vendor" instead of being deleted.

**Derived columns:** trip_minutes, pickup_date, pickup_hour, day_type, tip_percent, avg_speed_mph
**Decoded labels:** payment_method, vendor_name